# **Raw dataset construction**

We gather panel data (country/year) on wellbeing, inequality, material consumption and emissions from various official sources listed below:

## Data Sources

| Dataset | Coverage | Source | Purpose in Project | Original Format
|---|---|---|---|---|
| OWID CO₂ Data | Annual country-level data (varies by variable; broad coverage 1900–2023) | https://github.com/owid/co2-data | Base dataset | Long |
| OWID Energy Data | Annual country-level energy indicators | https://github.com/owid/energy-data | Supplementary energy transition indicators | Long |
| Material Footprint per Capita | 2012–2021 | https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country | Per-capita material consumption indicator | Wide |
| World Bank GINI Index | 1992–2025 (sparse by country) | https://data.worldbank.org/indicator/SI.POV.GINI | Income inequality indicator | Wide |
| World Happiness Index | 2013–2023 | https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023 | Subjective wellbeing indicator | Long |

## Temporal Overlap

The main datasets overlap consistently between **2013 and 2021**, giving a usable multi-year comparison window for merged analysis.

---

## Core Theoretical Variables from OWID CO₂ and Energy Datasets

| Variable | Description | Analytical Purpose |
|---|---|---|
| `population` | Total population of the country | Useful for weighting and validating per-capita indicators |
| `gdp` | Gross Domestic Product | Main macroeconomic control variable |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Key variable linking emissions to lifestyles and consumption patterns |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Used for comparison with consumption emissions and emissions-gap calculations |
| `renewables_share_energy` | Share of primary energy consumption from renewable sources | Indicator of energy transition and decarbonization |
| `energy_per_capita` | Primary energy consumption per capita | Proxy for energy intensity and resource use |

---

## Identifier and Merge Variables

| Variable | Description | Role in Merge |
|---|---|---|
| `iso_code` | ISO 3-letter country code | Main country-level merge key |
| `country` | Full country name | Human-readable country identifier |
| `year` | Observation year | Temporal merge key for panel structure |

---

## Conceptual Focus of the Project

This project investigates whether countries can achieve relatively high levels of wellbeing while maintaining lower levels of environmental and material throughput.

The analysis combines:
- wellbeing indicators,
- inequality measures,
- emissions data,
- renewable energy transition metrics,
- and material footprint indicators

to explore possible forms of:
- sustainable wellbeing,
- ecological efficiency,
- and partial decoupling between quality of life and material consumption.

### Strategy summary:
1. Import the raw datasets independently.
2. Convert wide data to long format.
3. Check temporal coverage per dataset.
4. Keep only overlapping window accross all datasets
5. Check and normalise merge keys accross datasets (iso, country, year).
6. From OWID CO2 and Energy data, pick selected columns to keep.
7. Merge by country (iso) and year keys to create the full raw dataset to be cleaned.

# 1. Import the necessary libraries

In [1]:
import pandas as pd

# 2. Load the data

In [2]:
# Long format data
owid_co2_df = pd.read_csv('../data/raw/_owid_co2_data.csv')
owid_energy_df = pd.read_csv('../data/raw/owid_energy_data.csv')
hi_df = pd.read_csv('../data/raw/world_happiness_index_data.csv')

# Wide format data has to be converted to long prior to any merging
mf_df_wide = pd.read_csv('../data/raw/material_footprint_data_wide.csv')
gini_df_wide = pd.read_csv('../data/raw/gini_data_wide.csv')



We double check long/wide data formats have been correctly identified, and look at dataset shapes.

In [3]:
print("OWID CO2 DataFrame shape:", owid_co2_df.shape)
display(owid_co2_df.head(3))

print("OWID Energy DataFrame shape:", owid_energy_df.shape)
display(owid_energy_df.head(3))

print("World Happiness Index DataFrame shape:", hi_df.shape)
display(hi_df.head(3))

print("Material Footprint DataFrame shape:", mf_df_wide.shape)
display(mf_df_wide.head(3))

print("Gini DataFrame shape:", gini_df_wide.shape)
display(gini_df_wide.head(3))

OWID CO2 DataFrame shape: (50411, 79)


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


OWID Energy DataFrame shape: (23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


World Happiness Index DataFrame shape: (1670, 4)


,Country,Year,Index,Rank
0,Afghanistan,2013,4.040,143.0
1,Afghanistan,2015,3.575,153.0
2,Afghanistan,2016,3.360,154.0


Material Footprint DataFrame shape: (195, 39)


,ISO3,Country,Continent,Hemisphere,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Material footprint per capita (tonnes) (1990),Material footprint per capita (tonnes) (1991),Material footprint per capita (tonnes) (1992),...,Material footprint per capita (tonnes) (2012),Material footprint per capita (tonnes) (2013),Material footprint per capita (tonnes) (2014),Material footprint per capita (tonnes) (2015),Material footprint per capita (tonnes) (2016),Material footprint per capita (tonnes) (2017),Material footprint per capita (tonnes) (2018),Material footprint per capita (tonnes) (2019),Material footprint per capita (tonnes) (2020),Material footprint per capita (tonnes) (2021)
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,2.33,2.28,2.35,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,2.44,2.66,4.67,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,6.63,5.91,5.65,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96


Gini DataFrame shape: (266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(mf_df_wide.columns)
print(gini_df_wide.columns)

Index(['ISO3', 'Country', 'Continent', 'Hemisphere',
       'Human Development Groups', 'UNDP Developing Regions',
       'HDI Rank (2021)', 'Material footprint per capita (tonnes) (1990)',
       'Material footprint per capita (tonnes) (1991)',
       'Material footprint per capita (tonnes) (1992)',
       'Material footprint per capita (tonnes) (1993)',
       'Material footprint per capita (tonnes) (1994)',
       'Material footprint per capita (tonnes) (1995)',
       'Material footprint per capita (tonnes) (1996)',
       'Material footprint per capita (tonnes) (1997)',
       'Material footprint per capita (tonnes) (1998)',
       'Material footprint per capita (tonnes) (1999)',
       'Material footprint per capita (tonnes) (2000)',
       'Material footprint per capita (tonnes) (2001)',
       'Material footprint per capita (tonnes) (2002)',
       'Material footprint per capita (tonnes) (2003)',
       'Material footprint per capita (tonnes) (2004)',
       'Material footprint

It makes sense before going wide-to-long, to standardize column names accross datasets. Doing this first saves time not only with wide datasets, but also ther unconventional column names in the World Happiness Index dataset.

A `clean_colum_names` function is created to accomodate this:

In [5]:
# Utility

import pandas as pd
import re


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize dataframe column names.

    Transformations:
    - lowercase
    - strip leading/trailing whitespace
    - replace spaces with underscores
    - remove parentheses
    - replace special characters with underscores
    - collapse repeated underscores

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.

    Returns
    -------
    pd.DataFrame
        Dataframe with cleaned column names.
    """

    cleaned_columns = []

    for col in df.columns:

        col = col.strip().lower()

        # Remove parentheses
        col = re.sub(r"[()]", "", col)

        # Replace non-alphanumeric characters with underscores
        col = re.sub(r"[^a-z0-9]+", "_", col)

        # Remove repeated underscores
        col = re.sub(r"_+", "_", col)

        # Remove leading/trailing underscores
        col = col.strip("_")

        cleaned_columns.append(col)

    df.columns = cleaned_columns

    return df

In [6]:
# Apply clean_column_names function and print the cleaned dataframes

owid_co2_df = clean_column_names(owid_co2_df)
owid_energy_df = clean_column_names(owid_energy_df)
hi_df = clean_column_names(hi_df)
mf_df_wide = clean_column_names(mf_df_wide)
gini_df_wide = clean_column_names(gini_df_wide)

display(owid_co2_df.head(3))
display(owid_energy_df.head(3))
display(hi_df.head(3))
display(mf_df_wide.head(3))
display(gini_df_wide.head(3))

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


,country,year,index,rank
0,Afghanistan,2013,4.040,143.0
1,Afghanistan,2015,3.575,153.0
2,Afghanistan,2016,3.360,154.0


,iso3,country,continent,hemisphere,human_development_groups,undp_developing_regions,hdi_rank_2021,material_footprint_per_capita_tonnes_1990,material_footprint_per_capita_tonnes_1991,material_footprint_per_capita_tonnes_1992,...,material_footprint_per_capita_tonnes_2012,material_footprint_per_capita_tonnes_2013,material_footprint_per_capita_tonnes_2014,material_footprint_per_capita_tonnes_2015,material_footprint_per_capita_tonnes_2016,material_footprint_per_capita_tonnes_2017,material_footprint_per_capita_tonnes_2018,material_footprint_per_capita_tonnes_2019,material_footprint_per_capita_tonnes_2020,material_footprint_per_capita_tonnes_2021
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,2.33,2.28,2.35,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,2.44,2.66,4.67,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,6.63,5.91,5.65,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96


,country_name,country_code,indicator_name,indicator_code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,unnamed_70
0,Aruba,ABW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We then manually change the remaining iso3 in mf_df_wide, and counry_name, country_code columns in gini_df_wide to match all other dataframes.

In [7]:
mf_df_wide = mf_df_wide.rename(columns={'iso3': 'iso_code'})
gini_df_wide = gini_df_wide.rename(columns={'country_name': 'country', 'country_code': 'iso_code'})

print(mf_df_wide.columns)
print(gini_df_wide.columns)

Index(['iso_code', 'country', 'continent', 'hemisphere',
       'human_development_groups', 'undp_developing_regions', 'hdi_rank_2021',
       'material_footprint_per_capita_tonnes_1990',
       'material_footprint_per_capita_tonnes_1991',
       'material_footprint_per_capita_tonnes_1992',
       'material_footprint_per_capita_tonnes_1993',
       'material_footprint_per_capita_tonnes_1994',
       'material_footprint_per_capita_tonnes_1995',
       'material_footprint_per_capita_tonnes_1996',
       'material_footprint_per_capita_tonnes_1997',
       'material_footprint_per_capita_tonnes_1998',
       'material_footprint_per_capita_tonnes_1999',
       'material_footprint_per_capita_tonnes_2000',
       'material_footprint_per_capita_tonnes_2001',
       'material_footprint_per_capita_tonnes_2002',
       'material_footprint_per_capita_tonnes_2003',
       'material_footprint_per_capita_tonnes_2004',
       'material_footprint_per_capita_tonnes_2005',
       'material_footprint_per_c

In [8]:
# Quick check - What is the 'unnamed_70' column in the gini_df_wide dataframe? It seems to be empty
# A: It's just an empty column - can drop.

print(gini_df_wide.shape)
print(f" Missing values in the unnamed_70 column: {gini_df_wide['unnamed_70'].isna().sum()}")

(266, 71)
 Missing values in the unnamed_70 column: 266


We now write a wide_to_long function that can be applied to both mf and gini DataFrames and gives us a long format with standardized column names for country, year and value.

In [9]:
import re
import pandas as pd


def wide_to_long(
    df: pd.DataFrame, id_vars: list, year_pattern: str, value_name: str
) -> pd.DataFrame:
    """Reshapes a dataframe from wide to long by identifying and renaming year
    columns internally using a regex pattern.

    Parameters
    ----------
    df : pd.DataFrame
        The wide dataframe.
    id_vars : list
        The identifier columns to keep.
    year_pattern : str
        Regex pattern containing a capture group () for the 4-digit year.
    value_name : str
        Name for the final metric column.
    """
    # 1. Map original column names to the extracted 4-digit year
    rename_map = {}
    for col in df.columns:
        match = re.search(year_pattern, col)
        if match:
            # Extract the year from the first capture group
            rename_map[col] = match.group(1)

    value_vars = list(rename_map.values())

    # 2. Rename columns and subset the dataframe
    # This automatically ignores/drops any columns not in id_vars or rename_map (like unnamed_70 in gini_df_wide)
    df_renamed = df.rename(columns=rename_map)[id_vars + value_vars]

    # 3. Melt the dataframe using the clean year strings
    df_long = pd.melt(
        df_renamed,
        id_vars=id_vars,
        value_vars=value_vars,
        var_name="year",
        value_name=value_name,
    )

    # 4. Convert year to integer
    df_long["year"] = df_long["year"].astype(int)

    return df_long

We now apply the function to both datasets, and check the resulting long format DataFrames.

In [10]:
mf_id_vars = [
    "iso_code",
    "country",
    "continent",
    "hemisphere",
    "human_development_groups",
    "undp_developing_regions",
    "hdi_rank_2021",
]

mf_df = wide_to_long(
    df=mf_df_wide,
    id_vars=mf_id_vars,
    year_pattern=r"material_footprint_per_capita_tonnes_(\d{4})",
    value_name="material_footprint_per_capita",
)

In [11]:
gini_id_vars = ["country", "iso_code", "indicator_name", "indicator_code"]

gini_df = wide_to_long(
    df=gini_df_wide,
    id_vars=gini_id_vars,
    year_pattern=r"^(\d{4})$",
    value_name="gini_index",
)


print(mf_df.shape)
print(gini_df.shape)
display(mf_df.head())
display(gini_df.head())

(6240, 9)
(17556, 6)


,iso_code,country,continent,hemisphere,human_development_groups,undp_developing_regions,hdi_rank_2021,year,material_footprint_per_capita
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,1990,2.33
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,1990,2.44
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,1990,6.63
3,AND,Andorra,Europe,Northern Hemisphere,Very High,NaN,40.0,1990,NaN
4,ARE,United Arab Emirates,Asia,Northern Hemisphere,Very High,AS,26.0,1990,64.75


,country,iso_code,indicator_name,indicator_code,year,gini_index
0,Aruba,ABW,Gini index,SI.POV.GINI,1960,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,1960,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,1960,NaN
3,Africa Western and Central,AFW,Gini index,SI.POV.GINI,1960,NaN
4,Angola,AGO,Gini index,SI.POV.GINI,1960,NaN


We must also deal with the Happiness Index dataset, where the column of interest has name "index", which is not comprehensive once we merge with other data. Also, this dataset does not contain the iso_code column we intend to use as the primary merge column later on.
1. We rename the index column to happiness_index.
2. We extract the iso_codes from the owid_co2_df and add them here to match other datasets.

In [12]:
# Step 1: Rename the 'index' column to 'happiness_index'
hi_df = hi_df.rename(columns={'index': 'happiness_index'})

Step 2 is more challenging and brings up a potential issue with the data: What is the country coverage of each dataset?

We write a function that takes two DataFrames, the names of their country columns, and a name for each and checks for missmatches in the countries that are covered by each.

In [13]:
import pandas as pd


def compare_values(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    col1: str,
    col2: str,
    name1: str = "Dataset 1",
    name2: str = "Dataset 2",
) -> dict:
    """
    Compare unique values between two dataframe columns.

    Returns values:
    - only in df1
    - only in df2
    - shared values
    """

    set1 = set(df1[col1].dropna())
    set2 = set(df2[col2].dropna())

    only_in_1 = sorted(set1 - set2)
    only_in_2 = sorted(set2 - set1)
    shared = sorted(set1 & set2)

    print(
        f"Unique values | {name1}: {len(set1)} | {name2}: {len(set2)}"
    )

    print(f"\nShared values: {len(shared)}")

    print(
        f"\n[1/2] In {name1} but missing from {name2} ({len(only_in_1)}):"
    )
    print(only_in_1 if only_in_1 else "None")

    print(
        f"\n[2/2] In {name2} but missing from {name1} ({len(only_in_2)}):"
    )
    print(only_in_2 if only_in_2 else "None")

    return {
        "shared": shared,
        "only_in_df1": only_in_1,
        "only_in_df2": only_in_2,
    }

In [14]:
# Compare Happiness Index vs OWID
compare_values(
    owid_co2_df,
    hi_df,
    col1="country",
    col2="country",
    name1="OWID CO2",
    name2="Happiness"
)

Unique values | OWID CO2: 254 | Happiness: 167

Shared values: 159

[1/2] In OWID CO2 but missing from Happiness (95):
['Africa', 'Africa (GCP)', 'Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Asia', 'Asia (GCP)', 'Asia (excl. China and India)', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Central America (GCP)', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Europe', 'Europe (GCP)', 'Europe (excl. EU-27)', 'Europe (excl. EU-28)', 'European Union (27)', 'European Union (28)', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'High-income countries', 'International aviation', 'International shipping', 'Kiribati', 'Kuwaiti Oil Fires', 'Kuwaiti Oil Fires (GCP)', 'Least developed countries (Jones et al.)', 'Liechtenstein', 'Low-income

{'shared': ['Afghanistan',
  'Albania',
  'Algeria',
  'Angola',
  'Argentina',
  'Armenia',
  'Australia',
  'Austria',
  'Azerbaijan',
  'Bahrain',
  'Bangladesh',
  'Belarus',
  'Belgium',
  'Belize',
  'Benin',
  'Bhutan',
  'Bolivia',
  'Bosnia and Herzegovina',
  'Botswana',
  'Brazil',
  'Bulgaria',
  'Burkina Faso',
  'Burundi',
  'Cambodia',
  'Cameroon',
  'Canada',
  'Central African Republic',
  'Chad',
  'Chile',
  'China',
  'Colombia',
  'Comoros',
  'Congo',
  'Costa Rica',
  'Croatia',
  'Cyprus',
  'Czechia',
  'Denmark',
  'Djibouti',
  'Dominican Republic',
  'Ecuador',
  'Egypt',
  'El Salvador',
  'Estonia',
  'Eswatini',
  'Ethiopia',
  'Finland',
  'France',
  'Gabon',
  'Gambia',
  'Georgia',
  'Germany',
  'Ghana',
  'Greece',
  'Guatemala',
  'Guinea',
  'Haiti',
  'Honduras',
  'Hong Kong',
  'Hungary',
  'Iceland',
  'India',
  'Indonesia',
  'Iran',
  'Iraq',
  'Ireland',
  'Israel',
  'Italy',
  'Jamaica',
  'Japan',
  'Jordan',
  'Kazakhstan',
  'Kenya',

This highlights two important and relevant facts about the dataset structures that we still need to deal with:
1. The same country may have different names, like Turkiye and Turkey.

    - *Note: this is only problematic in the happiness index data which doesnt have an iso_code column to start with*
    
2. The OWID CO2 dataset contains non country columns like Africa, Asia, Least developed countries etc.

In [15]:
# Which 'countries' values are missing iso_code in the OWID dataset?
# This also gives us a way to identify non-country identifiers in the OWID dataset which might be useful later.

missing_iso = owid_co2_df[
    owid_co2_df["iso_code"].isna() |
    (owid_co2_df["iso_code"] == "")
]

print(missing_iso["country"].unique())

<StringArray>
[                                  'Africa',
                             'Africa (GCP)',
                                     'Asia',
                               'Asia (GCP)',
             'Asia (excl. China and India)',
                    'Central America (GCP)',
                                   'Europe',
                             'Europe (GCP)',
                     'Europe (excl. EU-27)',
                     'Europe (excl. EU-28)',
                      'European Union (27)',
                      'European Union (28)',
                    'High-income countries',
                   'International aviation',
                   'International shipping',
                                   'Kosovo',
                        'Kuwaiti Oil Fires',
                  'Kuwaiti Oil Fires (GCP)',
 'Least developed countries (Jones et al.)',
                     'Low-income countries',
            'Lower-middle-income countries',
                        'Middle East (GCP

In [16]:
# Step 2: Grab iso_code column from owid_co2_df

# 1. Create a clean 1-to-1 mapping from the OWID dataset
# Drop duplicates since OWID has multiple rows per country and dropna since there are non-country entries which will not have an iso_code as confirmed by the code above
iso_mapping = owid_co2_df[["country", "iso_code"]].drop_duplicates().dropna()

missmatch = compare_values(
    iso_mapping,
    hi_df,
    col1="country",
    col2="country",
    name1="ISO_MAPPING",
    name2="Happiness"
)

Unique values | ISO_MAPPING: 218 | Happiness: 167

Shared values: 158

[1/2] In ISO_MAPPING but missing from Happiness (60):
['Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'Kiribati', 'Liechtenstein', 'Macao', 'Marshall Islands', 'Micronesia (country)', 'Monaco', 'Montserrat', 'Nauru', 'New Caledonia', 'Niue', 'North Korea', 'Palau', 'Papua New Guinea', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe', 'Seychelles', 'Sint Maarten (Dutch part)', 'Solomon Islands', 'Tonga', 'Tur

We write a function to check close matches between the identified countries that are only in either of the compared column-dataset pairs but not the other.

In [17]:
from difflib import get_close_matches

def check_close_matches(
    only_in_1,
    only_in_2,
    n_matches=3,
    cutoff=0.6
):
    """
    Find close string matches between two mismatch lists.

    Parameters
    ----------
    only_in_1 : list
        Values only found in dataset 1.

    only_in_2 : list
        Values only found in dataset 2.

    n_matches : int
        Maximum number of suggested matches.

    cutoff : float
        Similarity threshold between 0 and 1.
    """

    for value in only_in_1:

        matches = get_close_matches(
            value,
            only_in_2,
            n=n_matches,
            cutoff=cutoff
        )

        if matches:
            print(f"{value} -> {matches}")

In [18]:
check_close_matches(
    missmatch["only_in_df1"],
    missmatch["only_in_df2"]
)

North Korea -> ['North Cyprus']
Turkey -> ['Turkiye']


So the only country that is in both datasets but with different name is Turekey. We change this value manually to match the OWID dataset nomenclature (in English): 'Turkey' and check that Turkey now appears in the shared countries of the two datasets.

In [19]:
hi_df.loc[hi_df["country"] == "Turkiye", "country"] = "Turkey"

compare_values(
    iso_mapping,
    hi_df,
    col1="country",
    col2="country",
    name1="ISO_MAPPING",
    name2="Happiness"
)

Unique values | ISO_MAPPING: 218 | Happiness: 167

Shared values: 159

[1/2] In ISO_MAPPING but missing from Happiness (59):
['Andorra', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Bermuda', 'Bonaire Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei', 'Cape Verde', 'Christmas Island', 'Cook Islands', "Cote d'Ivoire", 'Cuba', 'Curacao', 'Democratic Republic of Congo', 'Dominica', 'East Timor', 'Equatorial Guinea', 'Eritrea', 'Faroe Islands', 'Fiji', 'French Polynesia', 'Greenland', 'Grenada', 'Guinea-Bissau', 'Guyana', 'Kiribati', 'Liechtenstein', 'Macao', 'Marshall Islands', 'Micronesia (country)', 'Monaco', 'Montserrat', 'Nauru', 'New Caledonia', 'Niue', 'North Korea', 'Palau', 'Papua New Guinea', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe', 'Seychelles', 'Sint Maarten (Dutch part)', 'Solomon Islands', 'Tonga', 'Tur

{'shared': ['Afghanistan',
  'Albania',
  'Algeria',
  'Angola',
  'Argentina',
  'Armenia',
  'Australia',
  'Austria',
  'Azerbaijan',
  'Bahrain',
  'Bangladesh',
  'Belarus',
  'Belgium',
  'Belize',
  'Benin',
  'Bhutan',
  'Bolivia',
  'Bosnia and Herzegovina',
  'Botswana',
  'Brazil',
  'Bulgaria',
  'Burkina Faso',
  'Burundi',
  'Cambodia',
  'Cameroon',
  'Canada',
  'Central African Republic',
  'Chad',
  'Chile',
  'China',
  'Colombia',
  'Comoros',
  'Congo',
  'Costa Rica',
  'Croatia',
  'Cyprus',
  'Czechia',
  'Denmark',
  'Djibouti',
  'Dominican Republic',
  'Ecuador',
  'Egypt',
  'El Salvador',
  'Estonia',
  'Eswatini',
  'Ethiopia',
  'Finland',
  'France',
  'Gabon',
  'Gambia',
  'Georgia',
  'Germany',
  'Ghana',
  'Greece',
  'Guatemala',
  'Guinea',
  'Haiti',
  'Honduras',
  'Hong Kong',
  'Hungary',
  'Iceland',
  'India',
  'Indonesia',
  'Iran',
  'Iraq',
  'Ireland',
  'Israel',
  'Italy',
  'Jamaica',
  'Japan',
  'Jordan',
  'Kazakhstan',
  'Kenya',

Now that country names are correctly matched, we give countries inthe happiness index dataset their corresponding ISO codes.

In [ ]:
# 2. Left merge the ISO codes into your Happiness Index dataframe
hi_df_iso = hi_df.merge(iso_mapping, on="country", how="left")

print(f"Out of {hi_df_iso[['country']].drop_duplicates().shape[0]} countries, {hi_df_iso[['iso_code']].drop_duplicates().dropna().shape[0]} have ISO codes.")

# Which countries dont have iso codes?
print(f"The countries that dont have iso_codes are: {hi_df_iso[hi_df_iso['iso_code'].isna()]['country'].unique()}")
display(hi_df_iso[['country', 'iso_code']].drop_duplicates())

Out of 167 countries, 159 have ISO codes.
The countries that dont have iso_codes are: <StringArray>
['Congo Brazzaville',    'Congo Kinshasa',       'Ivory Coast',
            'Kosovo',      'North Cyprus',       'Puerto Rico',
        'Somaliland',         'Swaziland']
Length: 8, dtype: str


,country,iso_code
0,Afghanistan,AFG
10,Albania,ALB
20,Algeria,DZA
30,Angola,AGO
40,Argentina,ARG
...,...,...
1620,Venezuela,VEN
1630,Vietnam,VNM
1640,Yemen,YEM
1650,Zambia,ZMB
